In [2]:
import os, re, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
import openml
from scipy.special import comb
from scipy.stats import spearmanr

RUNS_ACC = Path("../data/runs_acuracia")
pastas = sorted([p for p in RUNS_ACC.iterdir() if p.is_dir()], key=lambda p: int(p.name.split("_")[1]))

completas = [p for p in pastas if (p / "COMPLETA").exists()]
print("pastas de task:", len(pastas), "| com COMPLETA:", len(completas))
print("sem COMPLETA:", [p.name for p in pastas if p not in completas])
assert len(completas) == 60, "Aguarde as 60 tasks terminarem antes de consolidar."

t0 = time.time()
partes = []
for p in pastas:
    task_id = int(p.name.split("_")[1])
    paginas = sorted(p.glob("pagina_*.csv.gz"))
    d = pd.concat([pd.read_csv(f) for f in paginas], ignore_index=True)
    d.insert(0, "task_id", task_id)
    partes.append(d)

df = pd.concat(partes, ignore_index=True)
print(f"{len(df)} runs em {df.task_id.nunique()} tasks | {time.time()-t0:.0f}s")
print("duplicados (task, run_id):", df.duplicated(["task_id", "run_id"]).sum())
print("nulos:\n", df.isna().sum())
print("fora de [0, 1]:", ((df["value"] < 0) | (df["value"] > 1)).sum())

#Compara volume baixado com a referência de AUC
est = pd.read_csv("../data/contagem_runs.csv").rename(columns={"tid": "task_id"}).set_index("task_id")
cmp = est.join(df.groupby("task_id").size().rename("baixados_acc"))
cmp["dif"] = cmp["baixados_acc"] - cmp["runs_aprox"]
print("\nResumo da diferença (acurácia - ref. AUC):")
print(cmp["dif"].describe().round(1))

df.to_parquet("../data/runs_acuracia_consolidado.parquet", index=False)

pastas de task: 60 | com COMPLETA: 60
sem COMPLETA: []
3624529 runs em 60 tasks | 59s
duplicados (task, run_id): 0
nulos:
 task_id        0
run_id         0
setup_id       0
flow_id        0
value          0
upload_time    0
dtype: int64
fora de [0, 1]: 0

Resumo da diferença (acurácia - ref. AUC):
count     60.0
mean     492.2
std      288.6
min       24.0
25%      231.2
50%      490.0
75%      719.5
max      986.0
Name: dif, dtype: float64


In [3]:
#Identifica run_ids marcados como degenerados na limpeza do AUC
auc_bruto = pd.read_parquet("../data/runs_consolidado.parquet", columns=["run_id"])
auc_limpo = pd.read_parquet("../data/runs_limpos_v3.parquet", columns=["run_id"])

ids_auc_bruto = set(auc_bruto["run_id"])
ids_auc_deg   = ids_auc_bruto - set(auc_limpo["run_id"])

mask_auc_deg = df["run_id"].isin(ids_auc_deg)
em_comum     = df["run_id"].isin(ids_auc_bruto)
print(f"Runs também presentes no AUC: {em_comum.sum()} ({em_comum.mean()*100:.1f}%)")
print(f"Removidos via run_id degenerado no AUC: {mask_auc_deg.sum()} ({mask_auc_deg.mean()*100:.1f}%)")

#Para runs exclusivos da acurácia, busca picos modais perto da classe majoritária
tasks_meta = openml.tasks.list_tasks(task_id=sorted(df["task_id"].unique().tolist()), output_format="dataframe")
p_maj = (tasks_meta["MajorityClassSize"] / tasks_meta["NumberOfInstances"]).set_axis(tasks_meta["tid"])

d_tmp = df[~mask_auc_deg].copy()
excl  = d_tmp[~d_tmp["run_id"].isin(ids_auc_bruto)]
print(f"Runs exclusivos da acurácia para inspecionar: {len(excl)}")

mask_maj = pd.Series(False, index=d_tmp.index)
pares_maj = []
for tid, g in excl.groupby("task_id"):
    base = float(p_maj.loc[tid])
    n_task = (d_tmp["task_id"] == tid).sum()
    cand = g[g["value"].between(base - 0.02, base + 0.02)].assign(v4=lambda x: x["value"].round(4))
    if cand.empty:
        continue
    cnt = cand.groupby("v4").size()
    picos = cnt[(cnt >= 200) & (cnt / n_task >= 0.005)]
    if not picos.empty:
        idx = g.index
        vals = g["value"].to_numpy()
        m = np.zeros(len(vals), dtype=bool)
        for c in picos.index:
            m |= np.abs(vals - c) <= 0.0005
            pares_maj.append({"task_id": tid, "p_maj": round(base, 4), "v4": c, "n": int(picos.loc[c])})
        mask_maj.loc[idx] = m

print(f"Picos majoritários extras (exclusivos acurácia): {len(pares_maj)} | runs removidos: {int(mask_maj.sum())}")
if pares_maj:
    print(pd.DataFrame(pares_maj).head(15).to_string(index=False))

d_limpo = d_tmp[~mask_maj].copy()
print(f"\nTotal: {len(df)} -> {len(d_limpo)} ({(1 - len(d_limpo)/len(df))*100:.1f}% removidos)")
print("Menor n por task após limpeza:", d_limpo.groupby("task_id").size().min())

d_limpo.to_parquet("../data/runs_acuracia_limpos.parquet", index=False)

Runs também presentes no AUC: 3624525 (100.0%)
Removidos via run_id degenerado no AUC: 845808 (23.3%)
Runs exclusivos da acurácia para inspecionar: 4
Picos majoritários extras (exclusivos acurácia): 0 | runs removidos: 0

Total: 3624529 -> 2778721 (23.3% removidos)
Menor n por task após limpeza: 4679


In [4]:
flows = pd.read_csv("../data/flows.csv")
df_f = d_limpo.merge(flows, on="flow_id", how="left")
print("flows sem nome:", df_f["flow_name"].isna().sum())

df_f["biblioteca"] = df_f["flow_name"].str.extract(r"^([A-Za-z0-9_]+)\.")[0]

BUSCAS = {"RandomizedSearchCV", "GridSearchCV", "HalvingRandomSearchCV", "HalvingGridSearchCV"}
WRAP_BUSCA = {"filteredclassifier", "multisearch", "multischeme"}

def split_top(s):
    partes, depth, cur = [], 0, ""
    for ch in s:
        depth += (ch == "(") - (ch == ")")
        if ch == "," and depth == 0:
            partes.append(cur); cur = ""
        else:
            cur += ch
    partes.append(cur)
    return partes

def familia_sk(s):
    s = re.sub(r"\(\d+\)$", "", s)
    head = s.split("(")[0]
    cls = head.split(".")[-1]
    args = s[len(head) + 1:-1] if "(" in s else ""
    try:
        if cls in BUSCAS:
            return familia_sk(split_top(args)[0].split("=", 1)[1])
        if cls == "Pipeline":
            return familia_sk(split_top(args)[-1].split("=", 1)[1])
    except IndexError:
        pass
    return cls.replace("FKC_EigenPro", "FKCEigenPro")

def familia_weka(nome):
    n = re.sub(r"^weka\.", "", nome)
    n = re.sub(r"\(\d+\)$", "", n).rstrip(")").split(".")[-1].lower()
    toks = [t for t in n.split("_") if t]
    while toks and toks[0] in WRAP_BUSCA and len(toks) > 1:
        toks = toks[1:]
    return toks[0]

sk = df_f["biblioteca"].eq("sklearn")
df_f.loc[sk, "familia"] = "sklearn:" + df_f.loc[sk, "flow_name"].map(familia_sk)

mlr_ = df_f["biblioteca"].eq("mlr")
df_f.loc[mlr_, "familia"] = "mlr:" + (df_f.loc[mlr_, "flow_name"]
    .str.replace(r"^mlr\.classif\.", "", regex=True)
    .str.replace(r"\(\d+\)$", "", regex=True)
    .str.replace(r"\.(preproc|imputed|tuned)", "", regex=True)
    .str.replace(r"\.dummied$", "", regex=True))

wk = df_f["biblioteca"].eq("weka") | (
    df_f["biblioteca"].isna() & ~df_f["flow_name"].str.startswith("automlbenchmark_")
)
df_f.loc[wk, "familia"] = "weka:" + df_f.loc[wk, "flow_name"].map(familia_weka)

df_f = df_f[df_f["familia"].notna()].copy()
df_f.to_parquet("../data/runs_acuracia_familias.parquet", index=False)

#Confere cobertura das 22 famílias finais definidas no AUC
fams_auc = pd.read_csv("../data/familias.csv", index_col=0)
fams_finais = sorted(fams_auc[fams_auc["final"]].index.tolist())

cont = df_f.groupby(["task_id", "familia"]).size().unstack(fill_value=0)
cob_acc = (cont >= 3).mean()

print("Famílias com >= 70% (min 3 runs) na acurácia:", (cob_acc >= 0.70).sum())
print("\nCobertura das 22 famílias do AUC na acurácia:")
print(cob_acc.reindex(fams_finais).sort_values().round(3).to_string())

flows sem nome: 0
Famílias com >= 70% (min 3 runs) na acurácia: 22

Cobertura das 22 famílias do AUC na acurácia:
familia
sklearn:GaussianNB                        0.700
mlr:naiveBayes                            0.717
weka:lwl                                  0.717
weka:ibk                                  0.767
weka:attributeselectedclassifier          0.767
weka:bagging                              0.767
weka:multilayerperceptron                 0.767
weka:adaboostm1                           0.767
weka:reptree                              0.767
weka:naivebayes                           0.767
mlr:rpart                                 0.783
weka:j48                                  0.783
weka:logistic                             0.800
weka:randomforest                         0.833
sklearn:GradientBoostingClassifier        0.883
sklearn:LogisticRegression                0.983
sklearn:RandomForestClassifier            1.000
sklearn:SVC                               1.000
sklearn:FKCEig

In [5]:
def emax_exact(valores, k=3):
    v = np.sort(np.asarray(valores, dtype=float))
    n = len(v)
    if n < k:
        return np.nan
    i = np.arange(1, n + 1)
    return float((v * comb(i - 1, k - 1) / comb(n, k)).sum())

df22 = df_f[df_f["familia"].isin(fams_finais)]
g = df22.groupby(["task_id", "familia"])["value"]
cel = pd.DataFrame({"n": g.size(), "max": g.max(), "mediana": g.median()}).reset_index()
cel = cel[cel["n"] >= 3]

sub = (df22.groupby(["task_id", "familia"])["value"]
           .apply(lambda s: emax_exact(s, 3))
           .rename("max_sub3").reset_index())
cel = cel.merge(sub, on=["task_id", "familia"])

bases = sorted(df["task_id"].unique())
def pivot(col):
    return cel.pivot(index="task_id", columns="familia", values=col).reindex(index=bases, columns=fams_finais)

tab_sub3 = pivot("max_sub3")
tab_max  = pivot("max")
tab_med  = pivot("mediana")
tab_n    = pivot("n")

print("tab_sub3:", tab_sub3.shape, "| NaNs:", int(tab_sub3.isna().sum().sum()),
      f"({tab_sub3.isna().mean().mean():.1%})")

os.makedirs("../data/tabelas_acuracia", exist_ok=True)
for nome, t in {"tab_sub3": tab_sub3, "tab_max": tab_max, "tab_med": tab_med, "tab_n": tab_n}.items():
    t.to_parquet(f"../data/tabelas_acuracia/{nome}.parquet")
    t.to_csv(f"../data/tabelas_acuracia/{nome}.csv")
cel.to_parquet("../data/tabelas_acuracia/celulas_longa.parquet", index=False)

tab_sub3: (60, 22) | NaNs: 206 (15.6%)


In [6]:
#Núcleo alinhado com o do AUC (46 x 17)
nucleo_auc = pd.read_parquet("../data/tabelas/nucleo_sub3.parquet")
nucleo_acc = tab_sub3.reindex(index=nucleo_auc.index, columns=nucleo_auc.columns)
print("Núcleo acurácia:", nucleo_acc.shape, "| NaNs:", int(nucleo_acc.isna().sum().sum()))

if nucleo_acc.isna().sum().sum() == 0:
    nucleo_acc.to_parquet("../data/tabelas_acuracia/nucleo_sub3.parquet")
    nucleo_acc.to_csv("../data/tabelas_acuracia/nucleo_sub3.csv")

#Comparação de ranking (Spearman por base) entre AUC e Acurácia nas 60 bases
tab_auc = pd.read_parquet("../data/tabelas/tab_sub3.parquet")
corrs, top1_iguais = [], []
for tid in tab_auc.index:
    par = pd.concat([tab_auc.loc[tid].rename("auc"), tab_sub3.loc[tid].rename("acc")], axis=1).dropna()
    if len(par) >= 3:
        corrs.append(spearmanr(par["auc"], par["acc"])[0])
        top1_iguais.append(par["auc"].idxmax() == par["acc"].idxmax())

print("\nCorrelação de Spearman por base (AUC vs. Acurácia):")
print(pd.Series(corrs).describe().round(3))
print(f"Bases em que o top-1 é idêntico (AUC vs. Acurácia): {np.mean(top1_iguais)*100:.1f}%")

Núcleo acurácia: (46, 17) | NaNs: 0

Correlação de Spearman por base (AUC vs. Acurácia):
count    60.000
mean      0.695
std       0.177
min       0.261
25%       0.576
50%       0.724
75%       0.836
max       0.963
dtype: float64
Bases em que o top-1 é idêntico (AUC vs. Acurácia): 46.7%


## Encerramento do Bloco 1 — Consolidação da Meta-Base

Com a execução deste notebook e da extração das metacaracterísticas, encerra-se a etapa de **coleta, limpeza e estruturação dos dados (Bloco 1)**. Os três componentes da meta-base encontram-se persistidos e alinhados sobre as **60 bases selecionadas da suíte OpenML-CC18**:

1. **Meta-alvos em AUC-ROC (`data/tabelas/`):** derivados de **3.624.525** *runs* brutos (**2.778.717** após remoção de execuções degeneradas), consolidados em **22 famílias de classificadores** com cobertura $\ge 70\%$ (mínimo de 3 *runs* por célula) sob o estimador imparcial `max_sub3` ($E[\max]$ para $k=3$).
2. **Meta-alvos em Acurácia (`data/tabelas_acuracia/`):** derivados de **3.624.529** *runs* brutos (**2.778.721** após limpeza sincronizada por `run_id` e verificação de picos na classe majoritária), gerando a matriz completa `(60 × 22)` e o núcleo sem faltantes `(46 × 17)` espelhados ao AUC.
3. **Metacaracterísticas (`data/metacaracteristicas.csv`):** vetor de **72 descritores** por base (71 medidas determinísticas via `pymfe` nos grupos `general`, `statistical` e `info-theory`, somadas à fração global de valores ausentes `frac_missing`), sem valores infinitos ou colunas constantes.

### Síntese Comparativa (AUC vs. Acurácia)
* **Consistência das execuções:** 100% dos *runs* de AUC estão presentes na base de acurácia, permitindo comparar ambas as métricas sobre exatamente o mesmo conjunto de experimentos.
* **Divergência nos rankings:** a correlação de Spearman média entre AUC e Acurácia por base é de **0,695** (mediana **0,724**), e o classificador *top-1* coincide em apenas **46,7%** das bases — evidenciando que a escolha da métrica de avaliação altera substancialmente a recomendação ótima.